# validation check — KMeans (running) -> profitable clusters -> LR, DRAW market

`goals_foragst` space, draw market. One segmentation method: KMeans on the running season
features, keeping the clusters that beat the market on the training fold. The k-sweep at the
bottom is an extra check, outside the template.

In [16]:
# --- SETUP -------------------------------------------------------------------
GROUP = "major"   # league group: which <group>/ ABT to validate
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline

from evaluation.model_check import run_check, collect_bets, pooled_roi, roi_grouped, season_folds

abt = pd.read_parquet(f"C:/Users/twluc/frame/01_data/03_abt/{GROUP}/abt.parquet")

In [84]:
# --- SPACE (modifiable) — the model space, defined in one place --------------
# The MODEL space: the features the bet-signal model reads. This is the ONLY cell to change
# to move the model to another space. (The segmentation space is set in the SEGMENTER cell.)
FEATURES = [
    "hmt_season_goals_for_avg",
    "hmt_season_goals_agst_avg",
    "awt_season_goals_for_avg",
    "awt_season_goals_agst_avg", 
    'mrkt_draw_impl',
    'mrkt_entropy',
]

In [85]:
# --- TARGET (modifiable) -----------------------------------------------------
# OUTCOME is the 0/1 event we bet on. IMPLIED is the market-consensus implied probability
# (mrkt), used for both the profitability check and the ROI — one odds source, never mixed.
OUTCOME = "t_draw_flg"          # 1 when the match ended in a draw
IMPLIED = "mrkt_draw_impl"      # implied draw prob = 1 / mrkt draw odds (profitability + betting)

In [86]:
# --- ABT FILTERING -----------------------------------------------------------
# Keep matches where both teams have enough history for stable season averages; drop rows missing any
# feature or the target.
MIN_GAME_NUMBER = 8
matches = abt[
    (abt["hmt_game_number"] > MIN_GAME_NUMBER)
    & (abt["awt_game_number"] > MIN_GAME_NUMBER)
].dropna(subset=FEATURES + [IMPLIED, OUTCOME, "league", "season"]).copy()

print(f"matches: {len(matches)}   seasons: {sorted(matches['season'].unique())}   "
      f"leagues: {sorted(matches['league'].unique())}")

matches: 5533   seasons: ['2223', '2324', '2425', '2526']   leagues: ['england', 'france', 'germany', 'italy', 'spain']


In [87]:
# --- SEGMENTER (modifiable) — KMeans on the running space --------------------
# SEGMENT_FEATURES is the space the SEGMENTATION uses. It defaults to the model's FEATURES (segment on
# the same variables the model sees); override it with a different list to segment on other variables
# than the model — e.g. carve regions by strength/context while the model still reads goals. The model
# space (FEATURES, used by run_check below) is unaffected.
SEGMENT_FEATURES = [
    "hmt_season_goals_for_avg",
    "hmt_season_goals_agst_avg",
    "awt_season_goals_for_avg",
    "awt_season_goals_agst_avg",
]

# The method IS this notebook's identity; a different method = a different notebook. Here: fit KMeans on
# SEGMENT_FEATURES (train only), keep the clusters that beat the market on train, label each row by its
# kept cluster (NaN outside -> not bet). Contract: segmenter(train) -> labeler ; labeler(rows) -> Series.

# Keep the clusters where the target happened MORE often than the market priced it, on the TRAIN fold
# only, so the held-out fold never influences the choice.
def keep_profitable(train, labels, min_matches=100):
    frame = pd.DataFrame({"label": labels.to_numpy(),
                          "won": train[OUTCOME].to_numpy(),
                          "implied": train[IMPLIED].to_numpy()})
    stats = frame.groupby("label").agg(won=("won", "mean"), implied=("implied", "mean"), n=("won", "size"))
    return set(stats.index[(stats["n"] >= min_matches) & (stats["won"] > stats["implied"])])

def kmeans_profitable_segmenter(features, n_clusters, min_matches=100):
    def segmenter(train):
        km = KMeans(n_clusters=n_clusters, random_state=0, n_init=10).fit(train[features].to_numpy())
        kept = keep_profitable(train, pd.Series(km.predict(train[features].to_numpy()), index=train.index), min_matches)
        def labeler(rows):
            c = pd.Series(km.predict(rows[features].to_numpy()), index=rows.index)
            return c.where(c.isin(kept))     # non-kept clusters -> NaN -> not bet
        return labeler
    return segmenter

K = 5   # cluster count (see the k-sweep extra check below)
segmenter = kmeans_profitable_segmenter(SEGMENT_FEATURES, K)

# The segmenter's self-description: fit on all matches (in-sample) and show each cluster's edge — size,
# outcome rate vs implied, ROI, centre (in SEGMENT_FEATURES). In-sample: a map of WHERE the edge sits,
# not gated evidence. KEEP marks clusters a training fold would keep (positive edge).
_km = KMeans(n_clusters=K, random_state=0, n_init=10).fit(matches[SEGMENT_FEATURES].to_numpy())
_lab = _km.predict(matches[SEGMENT_FEATURES].to_numpy())
print(f"segment edges (in-sample, k={K})   centre = [{'  '.join(SEGMENT_FEATURES)}]")
print(" seg      n   rate%  impl%   edge     ROI   keep   centre")
for s in range(K):
    m = matches[_lab == s]
    dr, im = m[OUTCOME].mean(), m[IMPLIED].mean()
    centre = " ".join(f"{x:.2f}" for x in _km.cluster_centers_[s])
    print(f" {s:>3} {len(m):>7}  {dr*100:5.1f}  {im*100:5.1f}  {(dr-im)*100:+5.1f}  "
          f"{pooled_roi(m[IMPLIED], m[OUTCOME].astype(int)):+6.3f}  {'KEEP' if dr > im else '    '}  [{centre}]")

segment edges (in-sample, k=5)   centre = [hmt_season_goals_for_avg  hmt_season_goals_agst_avg  awt_season_goals_for_avg  awt_season_goals_agst_avg]
 seg      n   rate%  impl%   edge     ROI   keep   centre
   0    1053   29.6   28.3   +1.4  +0.048  KEEP  [1.04 1.92 1.35 1.48]
   1    1004   17.8   21.0   -3.2  -0.153        [2.17 1.06 1.37 1.42]
   2    1017   22.4   25.2   -2.8  -0.112        [1.35 1.45 2.16 1.06]
   3    1532   26.4   28.9   -2.5  -0.087        [1.19 1.23 1.19 1.23]
   4     927   28.0   24.9   +3.2  +0.128  KEEP  [1.39 1.43 1.02 1.95]


In [88]:
# --- MODEL (modifiable) ------------------------------------------------------
# The bet-signal model: any sklearn-style estimator with .fit(X, y) and .predict_proba(X). It is fit
# on the MODEL space (FEATURES) — independent of the segmentation space above.
def make_model():
    return make_pipeline(PolynomialFeatures(degree=2, include_bias=True),
                         StandardScaler(), LogisticRegression(max_iter=3000))

BUFFER = 0.02   # only bet when the model probability beats the implied one by at least this margin

## Summary
The three gates at a glance, and the one-line row written to `model_checks/results.md`.

In [89]:
_ = run_check(matches, features=FEATURES, outcome=OUTCOME, implied=IMPLIED,
              segmenter=segmenter, make_model=make_model, buffer=BUFFER,
              source="goals_foragst_draw_kmeans_match_lr", space="goals_foragst", target="draw",
              abt_filter="major, gn>8, dropna",
              segmentation=f"KMeans k={K} (match, running, {len(SEGMENT_FEATURES)}D)",
              segment_filter="train-profitable clusters")

goals_foragst_draw_kmeans_match_lr   pooled +0.180 (1016 bets)   [mrkt]

GATE 1  LOSO — ROI by test season           PASS  (+4/4 seasons)
    2223 +0.163    2324 +0.334    2425 +0.110    2526 +0.143

GATE 2  league — ROI by league / drop best   PASS
    england +0.250    france +0.236    germany +0.193    spain +0.173    italy +0.023
    pooled without best league (england): +0.160

GATE 3  walk-forward — ROI by test season    PASS
    2425 +0.144    2526 +0.143


## Extra checks — below the template
Everything above is the standard template (config cells + Summary). Cells below are notebook-specific
diagnostics, outside the template — they may use knobs a given segmenter has (e.g. a cluster count)
that other methods do not.

In [90]:
# EXTRA CHECK (below the template) — knob robustness: sweep the cluster count k = 2..8.
# Recomputes the whole honest per-fold pipeline for each k (segmenting on SEGMENT_FEATURES, model on
# FEATURES). A real edge survives a range of k; one that appears at a single k is fragile.
print(" k    pooled     bets   seasons+")
for k in range(2, 9):
    b = collect_bets(matches, season_folds(matches), features=FEATURES, outcome=OUTCOME, implied=IMPLIED,
                     segmenter=kmeans_profitable_segmenter(SEGMENT_FEATURES, k), make_model=make_model, buffer=BUFFER)
    ps = roi_grouped(b, "fold")
    print(f" {k:<3} {pooled_roi(b['implied'], b['won']):+.4f}   {len(b):>4}   +{int((ps > 0).sum())}/{len(ps)}")

 k    pooled     bets   seasons+
 2   -0.0636    235   +0/1
 3   -0.0184    175   +0/1
 4   +0.0921    620   +4/4
 5   +0.1803   1016   +4/4
 6   +0.1008    759   +4/4
 7   +0.1124    863   +4/4
 8   +0.0836    878   +4/4
